# FraudWatch — Mobile-wallet fraud detection

PaySim course project. Add your name and team. This notebook uses the shared Python modules beside it so the app and experiment use identical preprocessing. Results appear only after running on your CSV. No PaySim result is pre-filled.

## 1. Problem and decision
Predict `isFraud` before a transaction executes and flag suspicious transactions for review. Average precision compares ranking quality under imbalance; precision, recall and confusion counts describe the selected review policy. Cost units are hypothetical, not measured monetary losses.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown
from train import load_sample, split_time, run

DATA_PATH = Path('data/paysim.csv')
OUTPUT = Path('outputs')
MAX_ROWS = 300000
JOBS = 2
assert DATA_PATH.exists(), 'Put your extracted PaySim CSV in data/paysim.csv or change DATA_PATH.'

## 2. Data dictionary and sampling
Source: https://www.kaggle.com/datasets/ealaxi/paysim1

`step`: simulated hour. `type`: transaction type. `amount`: transaction amount in dataset units. `oldbalanceOrg` and `oldbalanceDest`: balances before execution. `isFraud`: 0 or 1. Account IDs, post-transaction balances and `isFlaggedFraud` are excluded.

The sampler assigns independent random priorities to all rows in chunks and retains the lowest priorities. It neither takes just the beginning of the file nor balances the target.

In [ ]:
sample, source_rows = load_sample(DATA_PATH, MAX_ROWS)
train, validation, test = split_time(sample)
print('Source rows:', source_rows, '| Sample rows:', len(sample))
display(train.dtypes.to_frame('dtype'))
display(train.describe(include='all'))
display(train.isna().sum().to_frame('missing'))
print('Training fraud prevalence:', train.isFraud.mean())

## 3. Exploratory analysis on training data only
Study distributions and associations before modeling. The held-out test must not guide feature selection. The chart below compares transaction type, class prevalence and the amount distribution. Correlations summarize multiple numeric features; they do not establish causation.

In [ ]:
from train import eda
OUTPUT.mkdir(exist_ok=True, parents=True)
insights = eda(train, OUTPUT)
display(Image(filename=str(OUTPUT / 'eda.png')))
display(Image(filename=str(OUTPUT / 'correlations.png')))
for line in insights:
    display(Markdown('- ' + line))

## 4. Preprocessing and temporal validation
Complete time steps split into 60% training, 20% validation and 20% test. These are proportions of distinct steps, not rows. Three expanding training folds keep every earlier step before its validation steps.

Pipeline: row-wise features, median imputation, standard scaling, one-hot encoding, classifier. Only training folds fit stateful transforms. Missing target/step/amount and invalid numeric values fail validation. Large amounts remain because tail observations may contain meaningful fraud. A ratio cap of 1e6 is explicit.

Review the implementation rather than treating it as a black box:

In [ ]:
import inspect
import core
import train as training
print(inspect.getsource(core.features))
print(inspect.getsource(training.pipeline))
print(inspect.getsource(training.split_time))
print(inspect.getsource(training.time_folds))

## 5. Baseline, three models and tuning
DummyClassifier predicts training prevalence. Logistic Regression and Random Forest use class weights. Histogram Gradient Boosting is also compared. Each algorithm searches two parameter settings across the same forward CV folds. This is a bounded search for a two-day project, not exhaustive tuning. The model with the highest mean CV average precision wins.

A separate validation period chooses a threshold minimizing assumed cost: 25 per missed fraud and 1 per false alarm. The model remains trained on the training period so its validation-selected threshold stays attached to the same model. Test outcomes never pick the winner.

The next cell performs the experiment once, or reads the existing run. To retrain, use a new OUTPUT folder. Keep the dataset and output folder paired.

In [ ]:
if (OUTPUT / 'report.json').exists():
    report = json.loads((OUTPUT / 'report.json').read_text(encoding='utf-8'))
    print('Displaying existing completed run. Change OUTPUT to train a new run.')
else:
    report = run(DATA_PATH, OUTPUT, MAX_ROWS, JOBS)
display(pd.read_csv(OUTPUT / 'cv_comparison.csv'))
print('Selected:', report['selected_model'])
print('Parameters:', report['best_params'])
print('Threshold:', report['threshold'])

## 6. Final untouched-test evaluation
The comparison table uses threshold 0.5 for every model. The selected-policy metrics below use the validation-selected threshold. Average precision is distinct from trapezoidal PR-AUC. Do not choose another model based on this table and reuse this test as unbiased evidence.

In [ ]:
display(pd.read_csv(OUTPUT / 'model_comparison.csv'))
display(pd.Series(report['test_selected']).to_frame('selected_policy'))
display(Image(filename=str(OUTPUT / 'precision_recall.png')))
display(Image(filename=str(OUTPUT / 'confusion_matrix.png')))

## 7. Error analysis
Inspect both missed fraud and false alarms. Describe their transaction types and amounts. Associations are not explanations of causal mechanisms. Record your own observations below.

In [ ]:
display(pd.read_csv(OUTPUT / 'errors_by_type.csv'))
display(pd.read_csv(OUTPUT / 'error_examples.csv').head(20))

**Complete after running:**
- Main false-alarm pattern: …
- Main missed-fraud pattern: …
- Operational implication and next experiment: …

## 8. Business recommendation and limitations
Use this as a human-review prioritization prototype. Quantify caught fraud, missed fraud and false alarms using this experiment. Do not convert hypothetical costs into claims of money saved.

PaySim is simulated. Scores are uncalibrated, sampling adds uncertainty and repeated accounts may cross time periods. Point metrics do not include confidence intervals. Temporal testing does not demonstrate new-account generalization. No real wallet deployment readiness is established.

## 9. Application and reproducibility
Run `python -m streamlit run app.py`. Review the single-transaction form and batch download. Threshold exploration uses validation scores only.

The outputs folder contains the fitted pipeline, recorded package versions, JSON report, charts, 12-slide presentation and LinkedIn draft. Add your name, measured interpretation, screenshot and deployed URL. Follow START_HERE_AR.md for deployment. Link the dataset instead of uploading it to GitHub.

In [ ]:
print(json.dumps(report['versions'], indent=2))
print('Generated files:')
for path in sorted(OUTPUT.iterdir()):
    print(path.name)